In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import sqlite3
import random
import ast
from estnltk import Text
from estnltk_neural.taggers import StanzaSyntaxTagger
from conllu.serializer import serialize_field
from estnltk.converters.conll.conll_exporter import layer_to_conll

### Setup

In [26]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
SOURCE_DATA = "verb_obj_cases_filtered.db"
RESULT_OLEMA = "verb_olema.conll"

### Methods

In [17]:
def get_table_data(db_path: str, table_name: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sentence_id, form_span, sentence FROM {table_name}".format(table_name=table_name))
    result = cursor.fetchall()
    conn.close()
    return result

In [27]:
stanza_tagger = StanzaSyntaxTagger()

In [38]:
def layer_to_custom_conll(
        text,
        layer,
        sentence_id,
        target_span,
        target_value,
        sentences_layer="sentences"
    ):

    CONLL_ATTRIBUTES = [
        "id",
        "form",
        "upostag",
        "head",
        "deprel"
    ]

    convertable_layer = text[layer]
    layer_has_form = "form" in convertable_layer.attributes

    output = []

    # Sentence ID line
    output.append(f"# sent_id = {sentence_id}")

    # There is exactly one sentence
    sentence = text[sentences_layer][0]

    for word in sentence:

        annotation = convertable_layer.get(
            word.base_span
        ).annotations[0]

        values = []

        for attr in CONLL_ATTRIBUTES:

            if attr == "form" and not layer_has_form:
                value = word.text
            else:
                value = annotation[attr]

            value = serialize_field(value)
            values.append(value)

        # Error column
        if (word.start, word.end) == target_span:
            error = target_value
        else:
            error = "_"

        values.append(serialize_field(error))

        output.append("\t".join(values))

    # Empty line after sentence
    output.append("")

    return "\n".join(output)

### I *Olema*-verb constructions

In [18]:
table_rows = get_table_data(f"{SOURCE_DATA_PATH}{SOURCE_DATA}", "verb_olema")

In [20]:
table_rows[0]

(2791587,
 '(0, 12)',
 'Üllatuspommi oli lõhkamas Kalevi eurosarjavastane Moskva Himki , juhtides suurfavoriidi Moskva CSKA vastu neli minutit enne lõppu 11 ja pool minutit enne sireeni 3 punktiga .')

In [22]:
random.shuffle(table_rows)

In [23]:
table_rows_100 = table_rows[:100]

In [25]:
table_rows_100[0]

(3740522,
 '(33, 39)',
 'Siimanni poolt hääletas 86 EOK-i liiget kohal olnud 99-st , teatas olümpiakomitee peasekretär Toomas Tõnise .')

In [39]:
for idx, row in enumerate(table_rows_100):
    sentence_text = Text(row[2]).tag_layer()
    sentence_stanza = stanza_tagger.tag(sentence_text)
    target_span = ast.literal_eval(row[1])
    print(target_span)
    conll_string = layer_to_custom_conll(
        text=sentence_text,
        layer="stanza_syntax",
        sentence_id=row[0],
        target_span=target_span,
        target_value="ERROR",
        sentences_layer="sentences"
    )
    with open(f"{SOURCE_DATA_PATH}{RESULT_OLEMA}", "a", encoding="UTF-8") as f:
        f.write(conll_string)
        f.write("\n")
    print(conll_string)

(33, 39)
# sent_id = 3740522
1	Siimanni	H	3	obl	_
2	poolt	K	1	case	_
3	hääletas	V	0	root	_
4	86	N	6	nummod	_
5	EOK-i	Y	6	nmod	_
6	liiget	S	8	obl	ERROR
7	kohal	S	8	obl	_
8	olnud	A	9	acl	_
9	99-st	N	3	nsubj	_
10	,	Z	11	punct	_
11	teatas	V	3	conj	_
12	olümpiakomitee	S	13	nmod	_
13	peasekretär	S	11	nsubj	_
14	Toomas	H	13	appos	_
15	Tõnise	H	14	flat	_
16	.	Z	3	punct	_

(41, 46)
# sent_id = 51230
1	Lihtsaima	H	2	nmod	_
2	näitena	S	3	obl	_
3	toob	V	0	root	_
4	ta	P	3	nsubj	_
5	suitsetamise	S	3	obj	_
6	,	Z	9	punct	_
7	mille	P	13	obj	ERROR
8	ta	P	9	nsubj	_
9	oleks	V	5	acl:relcl	_
10	nõus	D	9	compound:prt	_
11	päevapealt	D	13	advmod	_
12	maha	D	13	compound:prt	_
13	jätma	V	9	advcl	_
14	,	Z	17	punct	_
15	kui	J	17	mark	_
16	ainult	D	17	advmod	_
17	peaks	V	13	advcl	_
18	.	Z	3	punct	_

(41, 50)
# sent_id = 19010039
1	Täna	D	3	advmod	_
2	te	P	3	nsubj	_
3	selgitate	V	0	root	_
4	meile	P	3	nsubj	_
5	,	Z	9	punct	_
6	et	J	9	mark	_
7	õppekohtade	S	8	nmod	_
8	maksumust	S	9	nsubj	ERROR
9	oleks	V	3	ccomp	_
10	